In [ ]:
from openfermion.ops import QubitOperator
import numpy as np
from tqdm import tqdm
import pandas as pd
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import networkx as nx

In [ ]:
from qiskit.circuit.library import TwoLocal, RealAmplitudes, EfficientSU2
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit import Parameter
from qiskit_algorithms import NumPyMinimumEigensolver
# from qiskit import execute, Aer
from qiskit.primitives import BackendSampler
from qiskit.primitives import Sampler
from qiskit.providers.options import Options
# from qiskit.providers.aer import AerSimulator
from qiskit_algorithms.optimizers import SLSQP,COBYLA,ADAM, SPSA, QNSPSA,NFT
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import TwoLocal

In [ ]:
from qiskit_algorithms import SamplingVQE
from qiskit_aer import AerSimulator

In [ ]:
import matplotlib as mpl
import os

# Set TEX environment variable
# os.environ['TEX'] = '/home/himashu/.TinyTeX'

# Add TinyTeX bin directory to PATH
# os.environ['PATH'] = '/home/himashu/.TinyTeX/bin/x86_64-linux:' + os.environ['PATH']

# Configure matplotlib to use LaTeX
# mpl.rcParams['text.usetex'] = True
# plt.rcParams['font.family'] = 'comfortaa'
plt.rcParams['text.color'] = 'k'
plt.rcParams['axes.labelcolor'] = 'k'
# plt.style.use('seaborn')
# plt.style.use('default')

In [ ]:
adjacency_matrix = np.array([[0.,5.,3.,0.,0.,0.,0.,0.,0.,0.,0.],
 [0.,0.,5.,4.,0.,0.,0.,0.,0.,0.,0.],
 [0.,0.,0.,6.,4.,0.,0.,0.,0.,0.,0.],
 [0.,0.,0.,0.,5.,0.,0.,0.,0.,0.,0.],
 [0.,0.,0.,0.,0.,3.,0.,3.,0.,0.,0.],
 [0.,0.,0.,0.,0.,0.,6.,0.,0.,0.,0.],
 [0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,0.],
 [0.,0.,0.,0.,0.,0.,0.,0.,6.,4.,0.],
 [0.,0.,0.,0.,0.,0.,0.,0.,0.,5.,3.],
 [0.,0.,0.,0.,0.,0.,0.,0.,0.,0.,5.],
 [0.,0.,0.,0.,0.,3.,0.,3.,0.,0.,0.]],dtype = float)

In [ ]:
# Create a directed graph from the adjacency matrix
G = nx.from_numpy_array(adjacency_matrix, create_using=nx.DiGraph)

# Draw the graph
plt.figure(figsize=(6, 6))
pos = nx.circular_layout(G)  # positions for all nodes
nx.draw(G, pos, with_labels=True, node_color='skyblue', node_size=1500, edge_color='gray', arrowsize=20)
plt.title('Directed Graph from Adjacency Matrix')
# plt.savefig('graph.jpg')
plt.show()

In [ ]:
W = -10*np.array(adjacency_matrix,dtype = float)
num_pos = W.shape[0]
α = 1*np.max(abs(W))/10

for i in range(len(W)):
    for j in range(len(W)):
        if W[i,j] == 0:
            W[i,j] = α # + np.random.uniform()


node = W.shape[0]
node_qubit = int(np.ceil(np.log2(node)))
num_qubits = node_qubit*num_pos
print('Number of qubits for nodes :', node_qubit)
print('Number of positions : ',num_pos)
print('Number of qubits required:', num_qubits)

print(pd.DataFrame(W))

In [ ]:
def P0(i):
    iden = QubitOperator('')
    Z = QubitOperator('Z' + str(i))
    return 0.5*(iden + Z)


def P1(i):
    iden = QubitOperator('')
    Z = QubitOperator('Z' + str(i))
    return 0.5*(iden - Z)


def number_to_fixed_length_binary_array(number, length):
    binary_string = bin(number)[2:]  # Get binary representation as a string, removing the '0b' prefix
    padded_binary_string = binary_string.zfill(length)  # Pad with leading zeros to ensure fixed length
    binary_array = np.array([int(bit) for bit in padded_binary_string])  # Convert binary string to numpy array
    return binary_array

def occupation_op(num,pos,N):
    
    # num : node number 
    # N : total qubit for node
    # pos : position runs from 0 to edge - 1

    Ait = QubitOperator('')
    
    binary_array = number_to_fixed_length_binary_array(num, N)
    Nbit = len(binary_array)
    for i in range(Nbit):
        if binary_array[i] == 1:
            Ait = Ait*P1(i + pos*Nbit)
        else:
            Ait = Ait*P0(i + pos*Nbit)
   
    return Ait 


bit = []

for ti in (range(num_pos)):
    bi = []
    for i in range(node):
        bi.append(occupation_op(i,ti,node_qubit))
    bit.append(bi)

In [ ]:
# Visit Once 

identity = QubitOperator('')

H1 = 0 

A2 = 1.5*np.max(W)

for i in range(node):
    term = 0
    for t in range(num_pos):
        term += bit[t][i]
    term_sq = (identity-term)*(identity-term)
    H1 += term_sq

# Visit once version 02

# H1 = 0 

# for t in range(num_pos):
#     for tdash in range(t+1,num_pos):
#         for i in range(node):
#             H1 += 2*bit[t][i]*bit[tdash][i] - (1/(node-1))*(bit[t][i] + bit[tdash][i] + 2)


# including weights 

H2 = 0

for t in tqdm(range(num_pos - 1)):
    for i in range(node):
        for j in range(node):
            H2 += W[i,j]*(bit[t][i]*bit[t+1][j])

B = 1

H = A2*H1 + B*H2

In [ ]:
H_op = H.terms

print('Constructing Hamiltonian...Completed.')

print('Initiating VQE..')

op = []

for key, value in H_op.items():
    strlist = ['I' for i in range(num_qubits)]  
    for item in key:
        strlist[item[0]] = 'Z'
    op_string = ''.join(strlist)
    op.append((op_string,value))

# op.pop(0)

print(pd.DataFrame(op))

Hop = SparsePauliOp.from_list(op)

In [ ]:
## search for source node : 

for i in range(adjacency_matrix.shape[0]):
    x = np.sum(adjacency_matrix[:,i])
    if x == 0:
        print('source point -- ', i)
        source_node = i
        break


In [ ]:
## end node :

for i in range(adjacency_matrix.shape[0]):
    x = np.sum(adjacency_matrix[i,:])
    if x == 0:
        print('end point -- ', i)
        end_node = i
        break

In [ ]:
print('source node : ', source_node,' & end node : ', end_node)

In [ ]:
ref = QuantumCircuit(num_qubits)

# source_bit = number_to_fixed_length_binary_array(source_node,node_qubit)
# end_bit = number_to_fixed_length_binary_array(end_node,node_qubit)
# print(source_bit, end_bit)

In [ ]:
# ref = QuantumCircuit(num_qubits)

# source_bit = number_to_fixed_length_binary_array(source_node,node_qubit)
# end_bit = number_to_fixed_length_binary_array(end_node,node_qubit)

# for i in range(node_qubit):
#     if source_bit[i] == 1:
#         ref.x(num_qubits-1-i) 
#     if end_bit[i] == 1:
#         ref.x(i)

ref.draw()

In [ ]:
ansatz = QuantumCircuit(num_qubits)

for i in range(num_pos):
    # ansatz.append(EfficientSU2(node_qubit,su2_gates=["rx", "ry"], reps=1, parameter_prefix='φ' + str(i) , insert_barriers=True).decompose(), range(node_qubit + i*node_qubit,node_qubit + (i+1)*node_qubit))
    ansatz.append(EfficientSU2(node_qubit,su2_gates=["rx", "ry"], reps=1, parameter_prefix='φ' + str(i) , insert_barriers=True).decompose(), range(i*node_qubit,(i+1)*node_qubit))
ansatz = ref.compose(ansatz)

ansatz.measure_all()
ansatz.decompose().draw()

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    channel="ibm_cloud",
    token=" ",
    instance='cbp',
    set_as_default=True,
    # Use `overwrite=True` if you're updating your token.
    overwrite=True,
)
 
# Load saved credentials
service = QiskitRuntimeService()
backend = service.backend('ibm_torino')

In [ ]:
def get_qc_characteristics(qc):
    depth = qc.depth()
    num_qubits = qc.num_qubits
    ops = qc.count_ops()
    num_multi_qubit_ops = qc.num_nonlocal_gates()
    return {"depth": depth, "num_qubits": num_qubits, "ops": ops, "num_multi_qubit_ops": num_multi_qubit_ops}

def print_qc_characteristics(qc):
    characteristics = get_qc_characteristics(qc)
    print("Quantum circuit characteristics")
    print("  Depth:", characteristics['depth'])
    print("  Number of qubits:", characteristics['num_qubits'])
    print("  Operations:", dict(characteristics['ops']))
    print("  Number of multi-qubit Operations:", characteristics['num_multi_qubit_ops'])

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeTorino

from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

transpiled_circuit = generate_preset_pass_manager(3, backend=FakeTorino(), basis_gates=backend.operation_names, seed_transpiler=12345).run(ansatz.decompose())
print_qc_characteristics(transpiled_circuit)

In [ ]:
print_qc_characteristics(ansatz.decompose())

In [ ]:
print_qc_characteristics(transpiled_circuit)

In [ ]:
print(transpiled_circuit)

In [ ]:
import mthree

mapping = mthree.utils.final_measurement_mapping(transpiled_circuit)
mit = mthree.M3Mitigation(backend)
mit.cals_from_system(mapping)

In [ ]:
mit.cals_to_file('my_cals1.json')

In [ ]:
Hop = SparsePauliOp.from_list(op)
Hop_ = Hop.apply_layout(transpiled_circuit.layout)


import time
from qiskit.circuit.library import RealAmplitudes, EfficientSU2
from qiskit_algorithms.optimizers import COBYLA
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_algorithms.minimum_eigensolvers import SamplingVQE
#from qiskit_aer.primitives import Sampler, Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime import Session
from qiskit_aer import AerSimulator
from scipy.optimize import minimize
#from qiskit.result import sampled_expectation_value
from samplingVQECVAR import getExpectationValue


cost_history_dict = {
    "prev_vector": None,
    "iters": 0,
    "cost_history": [],
    "job_counts": [],
    "job_id": [],
}

params = np.array([2.853450287803932,  0.007712736382383185,  3.1128649383830003,  4.991335848649834,  6.648837933559497,  -6.723834830213132,  0.41138253740630937,  2.496804879734247,  -0.0598586284335238,  -2.705420872275853,  -9.648050707825021,  5.187683433752007,  -2.825959239274764,  3.0095937178382672,  9.643144315412165,  -0.6663476645747539,  -9.149807588935431,  0.36729592956966606,  -3.3334122717268433,  -6.066310515047025,  -7.156212281754527,  1.4166312732605113,  -1.2644530711931283,  -1.6290748670468624,  0.8383805786989746,  6.264386852097126,  4.272813334996502,  6.252839913210625,  5.012674476768021,  -4.5120296042475125,  -8.314532381417532,  -4.417935881730788,  2.9146748420754256,  2.879303746131755,  -2.902120892578933,  -1.3813378770741829,  4.586258507447635,  -5.4332105378714655,  9.44037564922223,  -8.82599519891333,  -6.189207970695925,  -1.5919332392653809,  -0.15399210302120853,  2.068793047505359,  -0.527133788469535,  -2.3469520750267017,  -9.83763138910896,  0.5684653426392774,  6.764767152061831,  -3.238229274305669,  -6.10590999413278,  6.64536927277719,  -7.731463065516211,  4.776422108337192,  -4.757361003648455,  3.8659428387788437,  -0.9052546158736321,  5.176882659564853,  1.68143095311546,  -4.443944158578239,  -1.8908063720583637,  4.966863009218281,  -4.659747092084887,  -1.7539152428022282,  -6.986795656715866,  -6.6034414398349845,  3.2263023917170552,  3.0964150337478613,  -0.9132082493889314,  7.548009177622017,  4.485942405869325,  1.4314862885627455,  -3.8512983526603732,  3.304113320519114,  -3.3153145109125997,  2.559975799163969,  -5.704549751045253,  1.827901454207883,  1.5500358075951763,  -7.953346058645108,  -6.586584274960305,  3.096772266003191,  6.501619169135251,  3.3161369046431974,  -3.587439543095107,  1.5511148604861442,  1.9136478682759506,  1.5794528451031251,  -0.23913871806591944,  -3.95685116919557,  -10.50330062218275,  -2.7399024959626304,  -2.880080586725549,  -1.417943817266666,  7.720449896385441,  5.271939107989121,  -6.602407917102591,  -3.825169979759312,  9.146902030779543,  9.368351684081457,  -2.7027623917127914,  7.363254493482785,  -1.8229103686501764,  1.8264012322088141,  9.47118265495385,  2.694137727742549,  -0.7393877334566813,  7.088177961828542,  3.4886287924334627,  -5.006683500945394,  -1.815168466903255,  -4.425808350829138,  -3.3724348511286446,  5.348703453716865,  0.10346199271865572,  -3.607697408437644,  6.913636167111887,  -6.663540041250135,  2.0045492685493262,  7.643346830169666,  6.060394195621187,  8.261946288603305,  9.060670940823297,  -4.244127499020441,  3.141461146422277,  5.80217521366013,  4.33810683970961,  -4.977520507849718,  7.019086332966733,  5.726177356573402,  0.16595259926791664,  0.32423921521857957,  4.86018948519888,  1.8577477496430732,  -1.3449296374204311,  4.710164875295494,  1.9344615081314596,  -0.155394771483625,  5.676571878215812,  3.199537979550589,  -7.397550627428301,  -4.888880096813974,  4.89713411492111,  -4.775888874889093,  0.342317963578019,  -3.1674751274603756,  -1.5306616678866898,  -6.414973131713344,  0.023074604196804495,  -3.095400804224139,  0.28651109678754405,  -1.3565646004734906,  3.083105366636651,  -9.140468411614371,  -4.582459923672269,  -2.4630204817942043,  3.1720611412802913,  -0.41412190684150396,  3.242223668505391,  -4.905023161856027,  -9.316995745640472,  -0.5031097231172923,  -2.668793770091361,  -6.964331620828338,  -12.844328383311765,  -5.175438869086996,  1.9269166715604509,  -7.615908911322679,  -2.6685855990883893,  1.0243007381718623,  -4.4668371672617315,  -7.134237056333896,  3.0914711397883186,  4.050695619099186,  4.527067338849573,  -4.273100157024597])
def cost_func(params, ansatz, hamiltonian, sampler):
    """Return estimate of energy from estimator

    Parameters:
        params (ndarray): Array of ansatz parameters
        ansatz (QuantumCircuit): Parameterized ansatz circuit
        hamiltonian (SparsePauliOp): Operator representation of Hamiltonian
        estimator (EstimatorV2): Estimator primitive instance
        cost_history_dict: Dictionary for storing intermediate results

    Returns:
        float: Energy estimate
    """
    #pub = (ansatz, [hamiltonian], [params])
    #result = estimator.run(pubs=[pub]).result()
    pub = (ansatz, [params])
    job = sampler.run([pub],shots=4096)
    result = job.result()
    samp_counts = result[0].data.meas.get_counts() #['__value__']['data'].meas.get_counts() #[0].data.meas.get_counts()
    quasis = mit.apply_correction(samp_counts, mapping)
    #energy = sampled_expectation_value(quasis, hamiltonian)
    energy = getExpectationValue([quasis], [hamiltonian], shots=4096, alpha=0.5)
    
    cost_history_dict["iters"] += 1
    cost_history_dict["prev_vector"] = params
    cost_history_dict["cost_history"].append(energy)
    cost_history_dict["job_counts"].append(list(quasis.keys()))
    print(f"Iters. done: {cost_history_dict['iters']} [Current cost: {energy}]")

    return energy

#with Session(backend= backend) as session:

sampler = Sampler(mode=backend)
sampler.options.twirling.enable_gates = True
sampler.options.twirling.enable_measure = False
sampler.options.dynamical_decoupling.enable = True
    
res = minimize(
    fun=cost_func,
    x0=params,
    bounds=[(-2*np.pi,2*np.pi)]*transpiled_circuit.num_parameters
    #args=(qc, Hop, estimator),
    args=(transpiled_circuit, Hop, sampler),
    method="COBYLA",options={"maxiter":50}
)


In [ ]:
len(list(job_result[0].data.meas.get_counts().keys())[0])

In [ ]:
import pickle
with open('costHist.pkl', 'wb') as f:
    pickle.dump(cost_history_dict, f)

In [ ]:
allbitS = []
for ii in range(len(cost_history_dict['job_counts'])):
    for jj in range(len(cost_history_dict['job_counts'][ii])):
        allbitS.append(cost_history_dict['job_counts'][ii][jj])

In [ ]:
len(allbitS)

In [ ]:
def violations(val):

    vio = 0
    num_seq = []

    for i in range(num_pos):
        num_seq.append(int(val[i*node_qubit:node_qubit*i+node_qubit],2))

    for i in range(num_pos-1):
        if ((num_seq[i] <= 10) and (num_seq[i+1] <= 10)):
            if adjacency_matrix[num_seq[i],num_seq[i+1]] == 0:
                vio +=1
                #print(num_seq[i],num_seq[i+1])
        else: vio += 100


    return vio,num_seq

#vio,num_seq = violations(val)

viols_ = []
#numseq_ = []
possibleSol = []
for ii in range(len(allbitS)):
    
    vio_,numseq = violations(allbitS[ii])
    if len(np.unique(numseq)) == len(numseq):
        viols_.append(vio_)
        possibleSol.append(allbitS[ii])

vio,num_seq = violations(possibleSol[np.argmin(np.array(viols_))])
print('No of violations ',vio)
print('Sequence ',num_seq)

In [ ]:
def list_of_all_connected_nodes(adjacency_mat:np.ndarray) -> list:
   
    """
    Takes adjacency matrix as input and 
    returns list of all connected nodes corresponding to every edge
    Useful to assess solutions
    """
    
    connected_nodes = [[i,j] for i in range(0,num_pos) for j in range (0,num_pos)
                        if adjacency_matrix[i][j]!=0]
    return connected_nodes

def connection_nodes(node:int, adjacency_mat:np.ndarray) -> list:

    """
    Takes a node and adjacency matrix as the input
    and returns list of nodes that have edges connecting them
    """

    connection_nodess = [j for j in range(0,num_pos) if adjacency_matrix[node][j]!=0]
    return connection_nodess

def violations(sol,adjacency_mat):

    """
    Takes solution and the adjacency matrix as inputs
    Returns list of nodes where violations are found
    """

    violation_sites = []
    for i in range(0,len(sol)-1):
        if sol[i+1] not in connection_nodes(sol[i],adjacency_mat):
            violation_sites.append(sol[i])
            print('Violation at Node',sol[i],'(Position',i,
                  '). Edge',[sol[i],sol[i+1]], 
                  'is not among', [[sol[i],j] for j in connection_nodes(sol[i],adjacency_mat)])
    return violation_sites

def repetitions(sol):

    """
    Takes solution candidate as the input
    Returns a list of the nodes in the solution that are repeated along the path 
    and their frequency
    """

    sol_set = list(set(sol))
    counts = [[i,sol.count(i)] for i in sol_set if sol.count(i)>1]
    [print('Node',counts[i][0],'repeated',counts[i][1],'times.') for i in range(0,len(counts))]
    return counts

def check_missing_nodes(sol,adjacency_mat):

    """
    Checks if there are any nodes missing in the path
    Takes the solution candidate and adjacency matrix as the input
    Returns a list of the missing nodes if they exist, else None 
    """

    missing_nodes = [i for i in range(0,len(adjacency_mat)) if i not in sol]
    if missing_nodes == []:
        print('No missing nodes in Solution')
        return None
    else:
        print('Solution missing node(s)',missing_nodes)
        return missing_nodes

def solution_quality_check(sol,adjacency_mat):

    """
    Checks the quality of solution by evaluating possible violations, repetitions,
    and missing nodes
    Takes solution candidate and adjacency matrix as the input
    Returns a dictionary with stats of violations, repetitions, and missing nodes
    """

    violation_counts = violations(sol,adjacency_mat)
    repetition_counts = repetitions(sol)
    missing_nodes = check_missing_nodes(sol,adjacency_mat)
    answer = {
            'violation_counts':[len(violation_counts),violation_counts],
            'repetition_counts':repetition_counts,
            'missing_nodes': missing_nodes
            }
    return answer

In [ ]:
# sol = [0, 1, 2, 3, 4, 7, 8, 9, 10, 5, 6]
solution_quality_check(num_seq,adjacency_matrix)

In [ ]:
list(raw_result.optimal_parameters.values())